# DEVOIR — CNN vs Transfert Learning (Cats vs Dogs)

## Objectif

Comparer un modèle **CNN entraîné from scratch** et un modèle en **transfert d'apprentissage**
(ResNet18 pré-entraîné sur ImageNet) sur le jeu de données Cats vs Dogs, et évaluer l'impact
du transfer learning sur la convergence, les métriques finales et le nombre de paramètres
entraînés.

**Plan du Travail :**
1. Setup & reproductibilité
2. Données : transformations, augmentation, split train/val/test
3. Fonctions d'entraînement et de métriques
4. Expérience A — CNN from scratch
5. Expérience B — Transfer Learning (ResNet18)
6. Comparaison des courbes d'apprentissage
7. Évaluation finale sur le jeu de test
8. Tableau récapitulatif
9. Conclusion

> Notebook pensé pour **Google Colab avec un runtime GPU**
> (`Exécution > Modifier le type d'exécution > GPU`).

## Accès aux données (Google Drive)

Montez votre Drive puis placez-vous dans le dossier contenant `Cat_Dog_data/`
(adaptez le chemin `%cd` à l'emplacement réel sur votre Drive).

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/DIT/deep_learning/devoir/Cat_Dog_data/
%ls


/content/drive/MyDrive/DIT/deep_learning/devoir/Cat_Dog_data
runs/  test/  train/


In [11]:
# Si Cat_Dog_data.zip n'est pas encore décompressé :
# !unzip -q Cat_Dog_data.zip


## 0. Setup & reproductibilité

In [5]:
import os
import random
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms, models

from sklearn.metrics import precision_score, recall_score, confusion_matrix
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device utilisé : {device}")
if device.type == 'cuda':
    print(f"GPU : {torch.cuda.get_device_name(0)}")
else:
    print("Aucun GPU détecté : pensez à activer un runtime GPU dans Colab "
          "(Exécution > Modifier le type d'exécution > GPU).")


Device utilisé : cuda
GPU : Tesla T4


## 1. Données : transformations, augmentation et split train/val/test

Le dossier `Cat_Dog_data` contient déjà `train/` et `test/`. On en extrait un jeu de
**validation** (10 % du train) pour suivre l'entraînement sans toucher au test final.
Les transformations d'entraînement incluent de la **data augmentation** raisonnable
(rotation, recadrage aléatoire, flip horizontal, léger jitter de couleur) ; validation et
test utilisent uniquement un resize + crop centré. La normalisation utilise les statistiques
ImageNet, nécessaires pour le modèle de transfer learning et réutilisées pour le CNN from
scratch afin de comparer les deux modèles sur des données identiques.

In [8]:
data_dir = '/content/drive/MyDrive/DIT/deep_learning/devoir/Cat_Dog_data/'
train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'test')

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Deux vues du même dossier train (une augmentée, une "propre") pour pouvoir appliquer des transforms différentes au sous-ensemble train et au sous-ensemble val.
train_dataset_aug = datasets.ImageFolder(train_dir, transform=train_transforms)
train_dataset_eval = datasets.ImageFolder(train_dir, transform=eval_transforms)
test_data = datasets.ImageFolder(test_dir, transform=eval_transforms)

class_names = train_dataset_aug.classes
print("Classes :", class_names)

VAL_FRACTION = 0.1
n_total = len(train_dataset_aug)
n_val = int(VAL_FRACTION * n_total)
n_train = n_total - n_val

split_generator = torch.Generator().manual_seed(SEED)
train_indices, val_indices = random_split(range(n_total), [n_train, n_val], generator=split_generator)

train_data = Subset(train_dataset_aug, train_indices.indices)
val_data = Subset(train_dataset_eval, val_indices.indices)

print(f"Train : {len(train_data)} | Val : {len(val_data)} | Test : {len(test_data)}")

BATCH_SIZE = 32
NUM_WORKERS = 2

trainloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))
valloader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))
testloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))


Classes : ['cat', 'dog']
Train : 20251 | Val : 2250 | Test : 2500


## 2. Fonctions d'entraînement et de métriques

`run_epoch` effectue une passe (train si un optimiseur est fourni, sinon évaluation) et
calcule **loss, accuracy, précision, rappel**. `train_model` boucle sur les époques, journalise
dans **TensorBoard** (`runs/<run_name>`), applique un **scheduler** optionnel, et conserve/
sauvegarde le **meilleur modèle** selon l'accuracy de validation (`checkpoints/<...>.pth`).

In [9]:
def run_epoch(model, loader, criterion, optimizer=None, desc=None):
    """Une passe sur `loader`. Entraîne si `optimizer` est fourni, sinon évalue.

    Affiche une barre de progression tqdm avec loss/accuracy courants,
    pour suivre l'entraînement batch par batch."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    running_correct = 0
    running_count = 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=desc, leave=False)
    with torch.set_grad_enabled(is_train):
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

            running_correct += (preds == labels).sum().item()
            running_count += batch_size
            pbar.set_postfix(loss=running_loss / running_count,
                              acc=running_correct / running_count)

    epoch_loss = running_loss / len(loader.dataset)
    accuracy = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)

    return {"loss": epoch_loss, "accuracy": accuracy, "precision": precision, "recall": recall}


def train_model(model, trainloader, valloader, optimizer, epochs, run_name,
                 criterion=None, scheduler=None, checkpoint_path=None, log_dir="runs"):
    criterion = criterion or nn.CrossEntropyLoss()
    writer = SummaryWriter(log_dir=os.path.join(log_dir, run_name))

    history = {"train": [], "val": []}
    best_val_acc = 0.0
    best_state = None

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_metrics = run_epoch(model, trainloader, criterion, optimizer,
                                   desc=f"[{run_name}] Époque {epoch}/{epochs} (train)")
        val_metrics = run_epoch(model, valloader, criterion, optimizer=None,
                                 desc=f"[{run_name}] Époque {epoch}/{epochs} (val)")

        if scheduler is not None:
            scheduler.step()

        history["train"].append(train_metrics)
        history["val"].append(val_metrics)

        for split, metrics in [("train", train_metrics), ("val", val_metrics)]:
            for k, v in metrics.items():
                writer.add_scalar(f"{k}/{split}", v, epoch)

        if val_metrics["accuracy"] > best_val_acc:
            best_val_acc = val_metrics["accuracy"]
            best_state = copy.deepcopy(model.state_dict())
            if checkpoint_path:
                torch.save(best_state, checkpoint_path)

        dt = time.time() - t0
        print(f"[{run_name}] Époque {epoch}/{epochs} ({dt:.1f}s) | "
              f"train_loss={train_metrics['loss']:.4f} acc={train_metrics['accuracy']:.3f} | "
              f"val_loss={val_metrics['loss']:.4f} acc={val_metrics['accuracy']:.3f} "
              f"prec={val_metrics['precision']:.3f} rec={val_metrics['recall']:.3f}")

    writer.close()
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_val_acc


## 3. Expérience A — CNN *from scratch*

Architecture simple à **4 blocs convolutifs** (`Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d`)
suivis d'un `AdaptiveAvgPool2d` et d'une tête dense avec **Dropout**.

- **BatchNorm** après chaque convolution : stabilise et accélère l'entraînement, permet
  d'utiliser un learning rate plus élevé.
- **Dropout** dans la tête dense (partie qui a le plus de paramètres donc le plus de risque
  de sur-apprentissage) : régularise en désactivant aléatoirement des neurones.

In [10]:
class CNNFromScratch(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            self._conv_block(3, 32),
            self._conv_block(32, 64),
            self._conv_block(64, 128),
            self._conv_block(128, 256),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    @staticmethod
    def _conv_block(in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


n_params = sum(p.numel() for p in CNNFromScratch().parameters() if p.requires_grad)
print(f"CNNFromScratch : {n_params:,} paramètres entraînables")


CNNFromScratch : 422,530 paramètres entraînables


### Recherche du meilleur optimiseur (Adam vs SGD)

Entraînement court (quelques époques) des deux optimiseurs pour comparer leur convergence
avant de lancer l'entraînement final plus long.

In [13]:
EPOCHS_SEARCH = 3

# La recherche d'optimiseur n'a besoin que de départager une tendance (Adam vs SGD),
# pas d'un entraînement complet : on la fait sur un sous-échantillon, ce qui la rend
# nettement plus rapide (tqdm affiche le nombre de batches réduit et l'ETA).
SEARCH_TRAIN_SIZE = 4000
SEARCH_VAL_SIZE = 800

search_generator = torch.Generator().manual_seed(SEED)
search_train_idx = torch.randperm(len(train_data), generator=search_generator)[:SEARCH_TRAIN_SIZE]
search_val_idx = torch.randperm(len(val_data), generator=search_generator)[:SEARCH_VAL_SIZE]

search_trainloader = DataLoader(Subset(train_data, search_train_idx), batch_size=BATCH_SIZE,
                                 shuffle=True, num_workers=NUM_WORKERS,
                                 pin_memory=(device.type == 'cuda'))
search_valloader = DataLoader(Subset(val_data, search_val_idx), batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=NUM_WORKERS,
                               pin_memory=(device.type == 'cuda'))

print(f"Recherche d'optimiseur sur {len(search_trainloader.dataset)} images train / "
      f"{len(search_valloader.dataset)} images val "
      f"(au lieu de {len(train_data)}/{len(val_data)} pour l'entraînement final)")

model_adam = CNNFromScratch().to(device)
opt_adam = optim.Adam(model_adam.parameters(), lr=1e-3)
model_adam, hist_adam, acc_adam = train_model(
    model_adam, search_trainloader, search_valloader, opt_adam, EPOCHS_SEARCH,
    run_name="cnn_from_scratch_adam_search")

model_sgd = CNNFromScratch().to(device)
opt_sgd = optim.SGD(model_sgd.parameters(), lr=1e-2, momentum=0.9)
model_sgd, hist_sgd, acc_sgd = train_model(
    model_sgd, search_trainloader, search_valloader, opt_sgd, EPOCHS_SEARCH,
    run_name="cnn_from_scratch_sgd_search")

print(f"Adam -> meilleure val acc : {acc_adam:.3f}")
print(f"SGD  -> meilleure val acc : {acc_sgd:.3f}")
BEST_OPTIMIZER = "adam" if acc_adam >= acc_sgd else "sgd"
print(f"Optimiseur retenu pour l'entraînement final : {BEST_OPTIMIZER}")


Recherche d'optimiseur sur 4000 images train / 800 images val (au lieu de 20251/2250 pour l'entraînement final)


[cnn_from_scratch_adam_search] Époque 1/3 (train):   0%|          | 0/125 [00:00<?, ?it/s]

[cnn_from_scratch_adam_search] Époque 1/3 (val):   0%|          | 0/25 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>assert self._parent_pid == os.getpid(), 'can only test a child process'

Traceback (most recent call last):
AssertionError  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    can only test a child processif w.is_alive():

  File "/usr/lib/

[cnn_from_scratch_adam_search] Époque 1/3 (466.2s) | train_loss=0.6698 acc=0.596 | val_loss=0.6481 acc=0.620 prec=0.622 rec=0.673


...

[cnn_from_scratch_sgd_search] Époque 2/3 (36.2s) | train_loss=0.6507 acc=0.617 | val_loss=0.6466 acc=0.632 prec=0.732 rec=0.455


[cnn_from_scratch_sgd_search] Époque 3/3 (train):   0%|          | 0/125 [00:00<?, ?it/s]

[cnn_from_scratch_sgd_search] Époque 3/3 (val):   0%|          | 0/25 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

[cnn_from_scratch_sgd_search] Époque 3/3 (37.6s) | train_loss=0.6404 acc=0.636 | val_loss=0.6442 acc=0.623 prec=0.785 rec=0.370
Adam -> meilleure val acc : 0.693
SGD  -> meilleure val acc : 0.632
Optimiseur retenu pour l'entraînement final : adam


### Entraînement final (plus d'époques + scheduler)

In [ ]:
EPOCHS_FINAL_A = 10

os.makedirs("checkpoints", exist_ok=True)

scratch_model = CNNFromScratch().to(device)
if BEST_OPTIMIZER == "adam":
    scratch_optimizer = optim.Adam(scratch_model.parameters(), lr=1e-3)
else:
    scratch_optimizer = optim.SGD(scratch_model.parameters(), lr=1e-2, momentum=0.9)
scratch_scheduler = optim.lr_scheduler.CosineAnnealingLR(scratch_optimizer, T_max=EPOCHS_FINAL_A)

scratch_model, history_scratch, best_acc_scratch = train_model(
    scratch_model, trainloader, valloader, scratch_optimizer, EPOCHS_FINAL_A,
    run_name="cnn_from_scratch", scheduler=scratch_scheduler,
    checkpoint_path="checkpoints/cnn_from_scratch_best.pth")

print(f"Meilleure accuracy validation (from scratch) : {best_acc_scratch:.3f}")


[cnn_from_scratch] Époque 1/10 (train):   0%|          | 0/633 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40><function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
        if w.is_alive():if w.is_alive():

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        assert self._parent_pid == os.getpid(), 'can only te

[cnn_from_scratch] Époque 1/10 (val):   0%|          | 0/71 [00:00<?, ?it/s]

[cnn_from_scratch] Époque 1/10 (1699.3s) | train_loss=0.6354 acc=0.634 | val_loss=0.6157 acc=0.654 prec=0.798 rec=0.421


[cnn_from_scratch] Époque 2/10 (train):   0%|          | 0/633 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>Traceback (most recent call last):

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()    
self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
if w.is_alive():  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        assert self._parent_pid == os.getpid(), 'can only te

[cnn_from_scratch] Époque 2/10 (val):   0%|          | 0/71 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40><function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>


  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Traceback (most recent call last):
Traceback (most recent call last):
Exception ignored in:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7b2e843fed40>        

self._shutdown_workers()self._shutdown_workers()Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_

[cnn_from_scratch] Époque 2/10 (167.2s) | train_loss=0.5875 acc=0.693 | val_loss=0.5325 acc=0.734 prec=0.727 rec=0.758


[cnn_from_scratch] Époque 3/10 (train):   0%|          | 0/633 [00:00<?, ?it/s]

## 4. Expérience B — Transfer Learning (ResNet18)

On repart d'un **ResNet18 pré-entraîné sur ImageNet** :
- Les couches convolutives précoces (features génériques : contours, textures) sont **gelées**.
- Le dernier bloc résiduel (`layer4`) est **dégelé** pour un léger *fine-tuning* sur nos images.
- La tête finale est remplacée par `Dropout -> Linear(2)`.

ResNet18 embarque déjà des `BatchNorm2d` dans chacun de ses blocs ; on y ajoute du **Dropout**
avant la couche de classification finale pour régulariser la partie ré-entraînée.

In [ ]:
def build_transfer_model(num_classes=2, dropout=0.4, unfreeze_last_block=True):
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    for param in model.parameters():
        param.requires_grad = False

    if unfreeze_last_block:
        for param in model.layer4.parameters():
            param.requires_grad = True

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


transfer_model = build_transfer_model().to(device)
trainable = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in transfer_model.parameters())
print(f"Transfer model : {trainable:,} / {total:,} paramètres entraînables")


In [ ]:
EPOCHS_B = 10

transfer_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, transfer_model.parameters()), lr=1e-4)
transfer_scheduler = optim.lr_scheduler.CosineAnnealingLR(transfer_optimizer, T_max=EPOCHS_B)

transfer_model, history_transfer, best_acc_transfer = train_model(
    transfer_model, trainloader, valloader, transfer_optimizer, EPOCHS_B,
    run_name="transfer_resnet18", scheduler=transfer_scheduler,
    checkpoint_path="checkpoints/transfer_resnet18_best.pth")

print(f"Meilleure accuracy validation (transfer learning) : {best_acc_transfer:.3f}")


## 5. Comparaison des courbes d'apprentissage

In [ ]:
def plot_history(histories, labels, metric):
    plt.figure(figsize=(7, 4))
    for history, label in zip(histories, labels):
        vals = [epoch_metrics[metric] for epoch_metrics in history["val"]]
        plt.plot(range(1, len(vals) + 1), vals, marker='o', label=label)
    plt.xlabel("Époque")
    plt.ylabel(metric.capitalize())
    plt.title(f"{metric.capitalize()} (validation) — comparaison des expériences")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


histories = [history_scratch, history_transfer]
labels = ["CNN from scratch", "Transfer learning (ResNet18)"]

for metric in ["loss", "accuracy", "precision", "recall"]:
    plot_history(histories, labels, metric)


## 6. Évaluation finale sur le jeu de test (rechargement des meilleurs modèles)

In [ ]:
def evaluate_on_test(model, checkpoint_path, name):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    metrics = run_epoch(model, testloader, nn.CrossEntropyLoss(), optimizer=None)
    print(f"[{name}] Test -> loss={metrics['loss']:.4f} acc={metrics['accuracy']:.3f} "
          f"precision={metrics['precision']:.3f} recall={metrics['recall']:.3f}")
    return model, metrics


scratch_model_final, test_metrics_scratch = evaluate_on_test(
    CNNFromScratch().to(device), "checkpoints/cnn_from_scratch_best.pth", "CNN from scratch")
transfer_model_final, test_metrics_transfer = evaluate_on_test(
    build_transfer_model().to(device), "checkpoints/transfer_resnet18_best.pth", "Transfer learning")


In [ ]:
def plot_confusion(model, loader, class_names, title):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds = model(images).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Prédiction")
    ax.set_ylabel("Vraie classe")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center', color=color)
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()


plot_confusion(scratch_model_final, testloader, class_names, "CNN from scratch — matrice de confusion")
plot_confusion(transfer_model_final, testloader, class_names, "Transfer learning — matrice de confusion")


## 7. Tableau récapitulatif

In [ ]:
summary = pd.DataFrame([
    {"Expérience": "CNN from scratch", **test_metrics_scratch},
    {"Expérience": "Transfer learning (ResNet18)", **test_metrics_transfer},
]).set_index("Expérience")

summary


## 8. Conclusion

_À compléter après exécution sur Colab :_
- Comparez la vitesse de convergence (nombre d'époques nécessaires) et le temps par époque.
- Comparez l'accuracy / précision / rappel finaux sur le test.
- Reliez les résultats au nombre de paramètres entraînables de chaque modèle.
- Expliquez pourquoi le transfer learning converge généralement plus vite et mieux sur un
  jeu de données de cette taille (features génériques déjà apprises sur ImageNet).
- Mentionnez les limites (temps de calcul, taille du jeu de validation, etc.) et des pistes
  d'amélioration (fine-tuning de plus de couches, augmentation plus poussée, autres backbones).